
# Building an STW-style Skin Tone Dataset — Acquisition, Stitching, EDA, Preprocessing

This notebook builds your own STW-style composite dataset (since the original STW
release isn't publicly downloadable yet), by:

1. Pulling in the public source datasets that STW itself is composed of
2. Unifying them into one manifest
3. Detecting/cropping faces so every image is on equal footing
4. Filtering for quality and de-duplicating
5. Auto-labeling skin tone via the ITA (Individual Typology Angle) method, mapped to
   the 10-point Monk scale
6. Running multi-graph EDA to sanity-check the result and check for bias
7. Writing a final `train/val/test/<monk_class>/` folder, ready for the training notebook

## Important — licensing / access reality check

Not all of STW's source datasets are freely auto-downloadable:

| Dataset | Access |
|---|---|
| LFW | Public, direct download |
| FEI Face Database | Public, direct download |
| Faces94/95 (Essex) | Public, direct download |
| CelebA | Public but hosted on Google Drive — download quotas are common; Kaggle mirror is more reliable |
| CASIA Face Africa / Face V5 | Requires a signed request to the CASIA team — **cannot be auto-downloaded** |
| FERET | Requires a signed license agreement via NIST — **cannot be auto-downloaded** |

This notebook automates what can be automated and gives you a clear manual-drop-in
point (`SOURCE_DIRS`) for the gated ones — download them yourself under their license
terms and point the config at wherever you put them.

## Important — labeling reality check

There's no shortcut to STW's real annotations — those came from trained human
annotators following a strict labeling protocol. What we build here is an **automated
proxy label** using the ITA (Individual Typology Angle) method, which is a documented,
peer-reviewed approach but is *not* equivalent to expert annotation. Treat it as a
starting point: spot-check a sample against the Monk scale swatches by eye before
trusting it, and consider manually correcting a subset.


In [ ]:

# %pip install facenet-pytorch opencv-python-headless imagehash pillow tqdm pandas numpy matplotlib seaborn scikit-learn requests -q


In [ ]:

import os
import io
import shutil
import hashlib
import zipfile
import tarfile
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import imagehash
import requests
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## Config

In [ ]:

WORK_DIR = Path("stw_build")
RAW_DIR = WORK_DIR / "01_raw"                 # downloaded/unpacked source datasets
FACES_DIR = WORK_DIR / "02_faces"              # after face detection + crop
CLEAN_DIR = WORK_DIR / "03_clean"              # after quality filter + dedup
FINAL_DIR = WORK_DIR / "04_final"              # final train/val/test/<class> layout
MANIFEST_DIR = WORK_DIR / "manifests"

for d in [RAW_DIR, FACES_DIR, CLEAN_DIR, FINAL_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Where each source dataset lives once downloaded/unpacked. Auto-downloadable ones
# get filled in by the download cells below; gated ones you fill in yourself.
SOURCE_DIRS = {
    "lfw": RAW_DIR / "lfw",
    "fei": RAW_DIR / "fei",
    "faces94_95": RAW_DIR / "faces94_95",
    "celeba": RAW_DIR / "celeba",              # fill after Kaggle/Drive download
    "casia_face_africa": RAW_DIR / "casia_face_africa",  # manual — gated
    "casia_face_v5": RAW_DIR / "casia_face_v5",           # manual — gated
    "feret": RAW_DIR / "feret",                            # manual — gated
}

MONK_CLASSES = [f"monk_{i:02d}" for i in range(1, 11)]
FACE_CROP_SIZE = 224
FACE_MARGIN = 0.35   # extra context around detected face box, as a fraction of box size

print("Working directory:", WORK_DIR.resolve())



## Step 1 — Acquire source datasets

Automated for the freely downloadable ones. For CelebA (Drive-quota-prone) and the
gated datasets, this cell tells you what to do rather than guessing at unstable URLs.


In [ ]:

def download_and_extract(url, dest_dir, archive_name=None, chunk_size=1 << 20):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    archive_name = archive_name or url.split("/")[-1]
    archive_path = dest_dir / archive_name

    if not archive_path.exists():
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(archive_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=archive_name) as pbar:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    f.write(chunk)
                    pbar.update(len(chunk))

    if archive_path.suffix == ".zip":
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(dest_dir)
    elif archive_path.suffixes[-2:] in ([".tar", ".gz"],) or archive_path.suffix == ".tgz":
        with tarfile.open(archive_path) as tf:
            tf.extractall(dest_dir)
    return dest_dir


In [ ]:

# --- LFW (public, direct) ---
try:
    download_and_extract(
        "https://vis-www.cs.umass.edu/lfw/lfw.tgz",
        SOURCE_DIRS["lfw"],
    )
    print("LFW ready at", SOURCE_DIRS["lfw"])
except Exception as e:
    print(f"LFW download failed ({e}) — download manually from https://vis-www.cs.umass.edu/lfw/ "
          f"and place under {SOURCE_DIRS['lfw']}")


In [ ]:

# --- CelebA ---
# Google Drive hosting makes this quota-prone from shared IPs (Colab included).
# Prefer the Kaggle mirror if you have a Kaggle API token configured:
#   from google.colab import files; files.upload()  # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d jessicali9530/celeba-dataset -p {SOURCE_DIRS['celeba']} --unzip
#
# Uncomment if you've set up the Kaggle API token:
# os.system(f"kaggle datasets download -d jessicali9530/celeba-dataset -p {SOURCE_DIRS['celeba']} --unzip")

if not any(SOURCE_DIRS["celeba"].glob("**/*.jpg")):
    print(f"⚠ CelebA not found under {SOURCE_DIRS['celeba']}. Download via Kaggle "
          "(see commented cell above) or https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html "
          "and place images under this path.")
else:
    print("CelebA ready at", SOURCE_DIRS["celeba"])


In [ ]:

# --- FEI Face Database (public, direct) ---
try:
    download_and_extract(
        "https://fei.edu.br/~cet/frontalimages_spatiallynormalized_part1.zip",
        SOURCE_DIRS["fei"],
        archive_name="fei_part1.zip",
    )
    download_and_extract(
        "https://fei.edu.br/~cet/frontalimages_spatiallynormalized_part2.zip",
        SOURCE_DIRS["fei"],
        archive_name="fei_part2.zip",
    )
    print("FEI ready at", SOURCE_DIRS["fei"])
except Exception as e:
    print(f"FEI download failed ({e}) — check current URL at https://fei.edu.br/~cet/facedatabase.html "
          f"(it moves occasionally) and place images under {SOURCE_DIRS['fei']}")


In [ ]:

# --- Faces94 / Faces95 (Essex, public) ---
print(
    "Faces94/95 don't have a single stable programmatic download URL — grab them from "
    "https://cswww.essex.ac.uk/mv/allfaces/index.html and place under "
    f"{SOURCE_DIRS['faces94_95']} (subfolders per identity, as distributed)."
)


In [ ]:

# --- Gated datasets: CASIA Face Africa, CASIA Face V5, FERET ---
gated = ["casia_face_africa", "casia_face_v5", "feret"]
for name in gated:
    p = SOURCE_DIRS[name]
    found = p.exists() and any(p.iterdir()) if p.exists() else False
    status = "found" if found else "NOT found — requires manual license/request, see notes above"
    print(f"{name}: {status} ({p})")



## Step 2 — Build a unified source manifest

Different datasets structure filenames/folders differently (per-identity subfolders,
flat files with identity encoded in the filename, etc). This normalizes them into one
table: `image_path, source_dataset, identity_id`.

Identity grouping matters for the later split — you don't want the same person's face
in both train and test.


In [ ]:

def scan_identity_folders(root, source_name, exts=(".jpg", ".jpeg", ".png", ".ppm", ".pgm")):
    '''For datasets laid out as root/<identity>/<image>.ext (LFW, CASIA, FEI-if-grouped).'''
    rows = []
    root = Path(root)
    if not root.exists():
        return rows
    for identity_dir in root.iterdir():
        if not identity_dir.is_dir():
            continue
        for img_path in identity_dir.rglob("*"):
            if img_path.suffix.lower() in exts:
                rows.append({
                    "image_path": str(img_path),
                    "source_dataset": source_name,
                    "identity_id": f"{source_name}_{identity_dir.name}",
                })
    return rows


def scan_flat_with_id_prefix(root, source_name, id_from_name, exts=(".jpg", ".jpeg", ".png")):
    '''For datasets laid out as root/<image>.ext where identity is encoded in the filename.'''
    rows = []
    root = Path(root)
    if not root.exists():
        return rows
    for img_path in root.rglob("*"):
        if img_path.suffix.lower() in exts:
            rows.append({
                "image_path": str(img_path),
                "source_dataset": source_name,
                "identity_id": f"{source_name}_{id_from_name(img_path.name)}",
            })
    return rows


all_rows = []
all_rows += scan_identity_folders(SOURCE_DIRS["lfw"] / "lfw", "lfw")
all_rows += scan_identity_folders(SOURCE_DIRS["casia_face_africa"], "casia_face_africa")
all_rows += scan_identity_folders(SOURCE_DIRS["casia_face_v5"], "casia_face_v5")
all_rows += scan_identity_folders(SOURCE_DIRS["faces94_95"], "faces94_95")

# FEI: filenames like "1-01.jpg" -> identity "1"
all_rows += scan_flat_with_id_prefix(
    SOURCE_DIRS["fei"], "fei", id_from_name=lambda n: n.split("-")[0]
)

# CelebA: use the official identity annotation file if present (identity_CelebA.txt),
# else fall back to treating every image as its own identity (weaker, but usable).
celeba_identity_file = SOURCE_DIRS["celeba"] / "identity_CelebA.txt"
if celeba_identity_file.exists():
    id_map = {}
    with open(celeba_identity_file) as f:
        for line in f:
            fname, ident = line.strip().split()
            id_map[fname] = ident
    for img_path in (SOURCE_DIRS["celeba"]).rglob("*.jpg"):
        ident = id_map.get(img_path.name, img_path.stem)
        all_rows.append({
            "image_path": str(img_path),
            "source_dataset": "celeba",
            "identity_id": f"celeba_{ident}",
        })
else:
    all_rows += scan_flat_with_id_prefix(
        SOURCE_DIRS["celeba"], "celeba", id_from_name=lambda n: Path(n).stem
    )

# FERET: typically identity encoded in filename prefix, e.g. "00001_930831_fa.ppm" -> "00001"
all_rows += scan_flat_with_id_prefix(
    SOURCE_DIRS["feret"], "feret", id_from_name=lambda n: n.split("_")[0]
)

source_manifest = pd.DataFrame(all_rows)
print(f"Total images found across sources: {len(source_manifest)}")
print(source_manifest["source_dataset"].value_counts())
source_manifest.to_csv(MANIFEST_DIR / "01_source_manifest.csv", index=False)



## Step 3 — Face detection & cropping

Every source dataset was shot under different conditions (pose, framing, background).
Standardize by detecting the face, cropping with a margin, and resizing — this also
gives the ITA skin-sampling step in Step 5 a consistent region to work with.


In [ ]:

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
# For better recall than Haar cascades, swap in facenet_pytorch's MTCNN if installed:
# from facenet_pytorch import MTCNN
# mtcnn = MTCNN(keep_all=False, device="cuda" if torch.cuda.is_available() else "cpu")

def detect_and_crop_face(image_path, out_size=FACE_CROP_SIZE, margin=FACE_MARGIN):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    if len(faces) == 0:
        return None

    # take the largest detected face
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    mx, my = int(w * margin), int(h * margin)
    x0, y0 = max(0, x - mx), max(0, y - my)
    x1, y1 = min(img.shape[1], x + w + mx), min(img.shape[0], y + h + my)
    crop = img[y0:y1, x0:x1]
    crop = cv2.resize(crop, (out_size, out_size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)


face_rows = []
for row in tqdm(source_manifest.itertuples(), total=len(source_manifest), desc="Detecting faces"):
    crop = detect_and_crop_face(row.image_path)
    if crop is None:
        continue
    out_path = FACES_DIR / row.source_dataset / f"{Path(row.image_path).stem}_{hashlib.md5(row.image_path.encode()).hexdigest()[:8]}.jpg"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(crop).save(out_path, quality=95)
    face_rows.append({
        "image_path": str(out_path),
        "source_dataset": row.source_dataset,
        "identity_id": row.identity_id,
    })

face_manifest = pd.DataFrame(face_rows)
detection_rate = len(face_manifest) / max(len(source_manifest), 1)
print(f"Face detected & cropped: {len(face_manifest)} / {len(source_manifest)} ({detection_rate:.1%})")
face_manifest.to_csv(MANIFEST_DIR / "02_face_manifest.csv", index=False)



## Step 4 — Quality filtering & deduplication

- Drop blurry images (Laplacian variance below threshold)
- Drop very low-resolution originals
- Drop near-duplicate images (perceptual hash) — datasets like LFW/CelebA can share
  crawled photos of the same public figures


In [ ]:

def blur_score(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.0
    return cv2.Laplacian(img, cv2.CV_64F).var()

def brightness_score(image_path):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return hsv[:, :, 2].mean()

BLUR_THRESHOLD = 30.0  # tune based on the histogram in the EDA section below

quality_rows = []
for row in tqdm(face_manifest.itertuples(), total=len(face_manifest), desc="Scoring quality"):
    b = blur_score(row.image_path)
    br = brightness_score(row.image_path)
    quality_rows.append({**row._asdict(), "blur_score": b, "brightness": br})

quality_df = pd.DataFrame(quality_rows).drop(columns=["Index"], errors="ignore")

# perceptual-hash dedup
hashes = {}
keep_mask = []
for path in tqdm(quality_df["image_path"], desc="Hashing for dedup"):
    h = str(imagehash.phash(Image.open(path)))
    if h in hashes:
        keep_mask.append(False)
    else:
        hashes[h] = path
        keep_mask.append(True)
quality_df["is_unique"] = keep_mask

clean_df = quality_df[
    (quality_df["blur_score"] >= BLUR_THRESHOLD) & (quality_df["is_unique"])
].reset_index(drop=True)

print(f"Before filtering: {len(quality_df)}")
print(f"After blur + dedup filtering: {len(clean_df)} "
      f"({len(quality_df) - len(clean_df)} removed)")

quality_df.to_csv(MANIFEST_DIR / "03_quality_scored_manifest.csv", index=False)



## Step 5 — Auto-label skin tone via ITA → Monk scale

The Individual Typology Angle (ITA) is computed from the L*a*b* color space of a
sampled skin patch:

```
ITA = arctan((L* - 50) / b*) × (180 / π)
```

Higher ITA → lighter skin, lower/negative ITA → darker skin. We sample a patch from
the center-face region (forehead/cheek area, avoiding eyes/mouth/hair) to reduce
non-skin contamination, then bucket ITA into 10 Monk-scale bins.

**This is an approximation** — the actual Monk scale is a perceptual swatch match, not
a pure colorimetric formula, and ITA thresholds-to-Monk mapping isn't standardized in
the literature. Treat the resulting labels as a strong starting point and spot-check
a random sample by eye against the official Monk swatches (https://skintone.google/the-scale)
before trusting them for training.


In [ ]:

def sample_skin_patch_lab(image_path, patch_frac=0.25):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    # Center patch, biased toward the cheek/forehead area (upper-middle of the crop)
    ph, pw = int(h * patch_frac), int(w * patch_frac)
    cy, cx = int(h * 0.38), int(w * 0.5)
    y0, y1 = max(0, cy - ph // 2), min(h, cy + ph // 2)
    x0, x1 = max(0, cx - pw // 2), min(w, cx + pw // 2)
    patch = img[y0:y1, x0:x1]
    if patch.size == 0:
        return None
    lab = cv2.cvtColor(patch, cv2.COLOR_BGR2LAB).astype(np.float32)
    # OpenCV LAB: L in [0,255] -> scale to [0,100]; a,b already centered near 128
    L = lab[:, :, 0].mean() * (100.0 / 255.0)
    a = lab[:, :, 1].mean() - 128.0
    b = lab[:, :, 2].mean() - 128.0
    return L, a, b


def compute_ita(L, b, eps=1e-6):
    return np.degrees(np.arctan((L - 50.0) / (b + eps)))


# ITA thresholds mapped to 10 Monk bins, ordered lightest (monk_01) to darkest (monk_10).
# These are evenly spaced across the typical ITA range (~55 to ~-30) as a starting
# heuristic — recalibrate against your own spot-checked samples.
ITA_BIN_EDGES = [55, 41, 28, 19, 10, 0, -10, -20, -30, -40, -100]

def ita_to_monk(ita_value):
    for i in range(len(ITA_BIN_EDGES) - 1):
        if ITA_BIN_EDGES[i] >= ita_value > ITA_BIN_EDGES[i + 1]:
            return MONK_CLASSES[i]
    return MONK_CLASSES[-1] if ita_value <= ITA_BIN_EDGES[-1] else MONK_CLASSES[0]


ita_rows = []
for row in tqdm(clean_df.itertuples(), total=len(clean_df), desc="Computing ITA"):
    patch = sample_skin_patch_lab(row.image_path)
    if patch is None:
        continue
    L, a, b = patch
    ita = compute_ita(L, b)
    ita_rows.append({
        "image_path": row.image_path,
        "source_dataset": row.source_dataset,
        "identity_id": row.identity_id,
        "L": L, "a": a, "b": b, "ita": ita,
        "monk_label": ita_to_monk(ita),
    })

labeled_df = pd.DataFrame(ita_rows)
labeled_df.to_csv(MANIFEST_DIR / "04_labeled_manifest.csv", index=False)
print(f"Labeled {len(labeled_df)} images")
labeled_df["monk_label"].value_counts().sort_index()


## Step 6 — Multi-graph EDA

In [ ]:

fig, ax = plt.subplots(figsize=(10, 4))
order = MONK_CLASSES
sns.countplot(data=labeled_df, x="monk_label", order=order, ax=ax, palette="rocket")
ax.set_title("Class distribution across Monk scale (auto-labeled)")
ax.set_xlabel("Monk class"); ax.set_ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(MANIFEST_DIR / "eda_class_distribution.png", dpi=150)
plt.show()


In [ ]:

# Bias check: does class distribution differ drastically by source dataset?
pivot = pd.crosstab(labeled_df["source_dataset"], labeled_df["monk_label"])
pivot = pivot.reindex(columns=MONK_CLASSES, fill_value=0)
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
pivot_norm.plot(kind="bar", stacked=True, ax=ax, colormap="rocket")
ax.set_title("Monk class composition per source dataset (row-normalized)")
ax.set_ylabel("Proportion")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(MANIFEST_DIR / "eda_source_vs_class.png", dpi=150)
plt.show()


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(labeled_df["ita"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("ITA value distribution")
axes[0].set_xlabel("ITA (degrees)")

sns.boxplot(data=labeled_df, x="monk_label", y="ita", order=MONK_CLASSES, ax=axes[1], palette="rocket")
axes[1].set_title("ITA by assigned Monk class (sanity check — should be monotonic)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(MANIFEST_DIR / "eda_ita_distribution.png", dpi=150)
plt.show()


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(quality_df["blur_score"], bins=40, ax=axes[0], color="darkorange")
axes[0].axvline(BLUR_THRESHOLD, color="red", linestyle="--", label=f"threshold={BLUR_THRESHOLD}")
axes[0].set_title("Blur score distribution (Laplacian variance)")
axes[0].legend()

sns.histplot(quality_df["brightness"].dropna(), bins=40, ax=axes[1], color="seagreen")
axes[1].set_title("Brightness distribution")
plt.tight_layout()
plt.savefig(MANIFEST_DIR / "eda_quality_distribution.png", dpi=150)
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(6, 5))
numeric_cols = labeled_df[["L", "a", "b", "ita"]].copy()
numeric_cols = numeric_cols.join(
    quality_df.set_index("image_path")[["blur_score", "brightness"]],
    on="image_path" if "image_path" in labeled_df.columns else None,
) if False else numeric_cols  # kept simple; join skipped to avoid index mismatch after filtering
corr = labeled_df[["L", "a", "b", "ita"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation between LAB color components and ITA")
plt.tight_layout()
plt.savefig(MANIFEST_DIR / "eda_correlation_heatmap.png", dpi=150)
plt.show()


In [ ]:

# Visual spot-check grid — actually look at a handful of images per class to validate
# the auto-labeling before trusting it.
fig, axes = plt.subplots(len(MONK_CLASSES), 5, figsize=(12, 2.2 * len(MONK_CLASSES)))
for row_idx, cls in enumerate(MONK_CLASSES):
    subset = labeled_df[labeled_df["monk_label"] == cls]
    sample_paths = subset["image_path"].sample(min(5, len(subset)), random_state=RANDOM_SEED).tolist()
    for col_idx in range(5):
        ax = axes[row_idx, col_idx]
        ax.axis("off")
        if col_idx < len(sample_paths):
            img = Image.open(sample_paths[col_idx])
            ax.imshow(img)
        if col_idx == 0:
            ax.set_ylabel(cls, fontsize=9)
plt.suptitle("Spot-check grid — verify these actually look like their assigned Monk class", y=1.001)
plt.tight_layout()
plt.savefig(MANIFEST_DIR / "eda_spotcheck_grid.png", dpi=150)
plt.show()



## Step 7 — Identity-aware stratified split

Split by `identity_id`, not by image — otherwise the same person can leak across
train/val/test and inflate your validation numbers.


In [ ]:

from sklearn.model_selection import train_test_split

# One row per identity, with a majority-vote class label (most identities in these
# source datasets have multiple photos, occasionally spanning adjacent auto-labels).
identity_labels = (
    labeled_df.groupby("identity_id")["monk_label"]
    .agg(lambda s: s.value_counts().idxmax())
    .reset_index()
)

train_ids, temp_ids = train_test_split(
    identity_labels, test_size=0.30, stratify=identity_labels["monk_label"], random_state=RANDOM_SEED
)
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, stratify=temp_ids["monk_label"], random_state=RANDOM_SEED
)

split_map = {}
split_map.update({i: "train" for i in train_ids["identity_id"]})
split_map.update({i: "val" for i in val_ids["identity_id"]})
split_map.update({i: "test" for i in test_ids["identity_id"]})

labeled_df["split"] = labeled_df["identity_id"].map(split_map)
print(labeled_df.groupby("split")["monk_label"].value_counts().unstack(fill_value=0))


In [ ]:

fig, ax = plt.subplots(figsize=(10, 4))
split_counts = labeled_df.groupby(["split", "monk_label"]).size().unstack(fill_value=0).reindex(columns=MONK_CLASSES)
split_counts.T.plot(kind="bar", ax=ax)
ax.set_title("Class balance across train/val/test splits")
ax.set_xlabel("Monk class"); ax.set_ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(MANIFEST_DIR / "eda_split_balance.png", dpi=150)
plt.show()


## Step 8 — Stitch final `train/val/test/<monk_class>/` layout

In [ ]:

for _, row in tqdm(labeled_df.iterrows(), total=len(labeled_df), desc="Writing final layout"):
    dest_dir = FINAL_DIR / row["split"] / row["monk_label"]
    dest_dir.mkdir(parents=True, exist_ok=True)
    src = Path(row["image_path"])
    dest = dest_dir / src.name
    if not dest.exists():
        try:
            os.symlink(src.resolve(), dest)
        except OSError:
            shutil.copy2(src, dest)

labeled_df.to_csv(MANIFEST_DIR / "05_final_manifest.csv", index=False)
print(f"Final dataset written to {FINAL_DIR}")
for split in ["train", "val", "test"]:
    n = sum(len(list((FINAL_DIR / split / c).glob("*"))) for c in MONK_CLASSES if (FINAL_DIR / split / c).exists())
    print(f"  {split}: {n} images")



## Summary

- `stw_build/manifests/05_final_manifest.csv` — full manifest with source, identity,
  LAB/ITA values, assigned Monk label, and split assignment, for full traceability.
- `stw_build/04_final/` — the `train/val/test/<monk_class>/` folder, ready to hand
  straight to the training notebook (`data_dir = "stw_build/04_final"`).
- `stw_build/manifests/eda_*.png` — the EDA charts, worth keeping alongside the
  dataset as documentation of known composition/bias for anyone using it later.

**Before training on this**: actually look at the spot-check grid above. If a chunk
of a class is visibly mislabeled, tighten the `ITA_BIN_EDGES` thresholds, improve the
skin-patch sampling region, or hand-correct a sample of borderline cases — the ITA
auto-labeling is a starting point, not ground truth.
